In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpdaf.obj import Cube
from astropy.coordinates import SkyCoord
import sys
import os
import plotfancy as pf
from matplotlib.patches import Circle
from astropy.visualization import ZScaleInterval
from astropy.table import Table
from types import SimpleNamespace
import re
from astropy.io import ascii
from astropy import units as u
from astropy.table import Table, vstack, hstack
from scipy.optimize import curve_fit
from astropy.cosmology import WMAP9 as cosmo
from astropy import coordinates as coords
from astroquery.sdss import SDSS
from requests.exceptions import ConnectionError
from matplotlib.lines import Line2D
from hst_phot import *

# from calculate_jiang19_metallicity import calculate_metallicity_jiang19 as cjm19
sys.path.append('../../')
import src.ifu_tools.line_ratios as lr

import logging 
logging.getLogger('mpdaf').setLevel(logging.WARNING)

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

pf.housestyle_rcparams()

In [ ]:
rest_lambdas = {
# --- Primary [OIII] and [OII] ---
'oiii5007': 5006.84,
'oiii4959': 4958.91,
'oii3726':  3726.03,
'oii3729':  3728.82,

# --- Hydrogen Balmer Series ---
'halpha':   6562.80,
'hbeta':    4861.33,
'hgamma':   4340.46,
'hdelta':   4101.73,
'hepsilon': 3970.08,
'hzeta':    3889.06,
'heta':     3835.40,

# --- Key Diagnostic Lines ---
'oiii4363': 4363.21,
'neiii':    3868.75,

# --- Low-Ionization Lines ---
'nii6583':  6583.45,
'nii6548':  6548.05,
'sii6716':  6716.44,
'sii6731':  6730.82,

# --- Helium Lines ---
'heii4686': 4685.68,
'hei5876':  5875.62,
}
balmer_lambda = {
# 'halpha':   6562.80,
'hbeta':    4861.33,
'hgamma':   4340.46,
'hdelta':   4101.73,
'hepsilon': 3970.08,
'hzeta':    3889.06,
'heta':     3835.40,
}
lambda_keys = {
# --- Primary [OIII] and [OII] ---
'oiii5007': r'[OIII] $\lambda$5007',
'oiii4959': r'[OIII] $\lambda$4959',
'oii3726':  r'[OII] $\lambda$3726',
'oii3729':  r'[OII] $\lambda$3729',

# --- Hydrogen Balmer Series ---
'halpha':   r'H$\alpha$',
'hbeta':    r'H$\beta$',
'hgamma':   r'H$\gamma$',
'hdelta':   r'H$\delta$',
'hepsilon': r'H$\epsilon$',
'hzeta':    r'H$\zeta$',
'heta':     r'H$\eta$',

# --- Key Diagnostic Lines ---
'oiii4363': r'[OIII] $\lambda$4363',  # Auroral line
'neiii':    r'[NeIII] $\lambda$3869',

# --- Low-Ionization Lines ---
'nii6583':  r'[NII] $\lambda$6583',
'nii6548':  r'[NII] $\lambda$6548',
'sii6716':  r'[SII] $\lambda$6716',
'sii6731':  r'[SII] $\lambda$6731',

# --- Helium Lines (not forbidden) ---
'heii4686': r'HeII $\lambda$4686',
'hei5876':  r'HeI $\lambda$5876',
}

In [ ]:
test = ascii.read('allsources.csv')
tab = test[test['object_id']!= 'STACK']
f = False
if f:
    fixed = 'fixed'
    tab['oiii4959_flux']=tab['oiii5007_flux']/3
else:
    fixed=''

In [ ]:
tab

In [ ]:
rat = tab['oii3729_flux']/tab['oii3726_flux']
mask = np.isfinite(rat)
plt.hist(rat[mask], bins=np.linspace(0,2.5,20), density=True)

In [ ]:
rat = tab['oiii5007_flux']/tab['oiii4959_flux']
mask1 = np.isfinite(rat)
plt.hist(rat[mask1], bins=np.linspace(0,10,20), density=True)

In [ ]:
R23 = (tab['oiii5007_flux'] + tab['oiii4959_flux'] +  tab['oii3729_flux'] + tab['oii3726_flux'])/tab['hbeta_flux']
mask2 = np.isfinite(R23)
plt.hist(np.log10(R23[mask2]), density=True, bins=np.linspace(0,2,15))

In [ ]:
O32 = (tab['oiii5007_flux'] + tab['oiii4959_flux'])/((tab['oii3729_flux'] + tab['oii3726_flux']))
O32_s = (tab['oiii5007_flux'])/(tab['oii3729_flux'])
mask3 = np.isfinite(O32)
mask_s = np.isfinite(O32_s)
plt.hist(np.log10(O32[mask3]), density=True, bins=np.linspace(-2,2,15), label=r'o32')
plt.hist(np.log10(O32_s[mask_s]), density=True, bins=np.linspace(-2,2,15), label=r'o32-s')
plt.legend()

In [ ]:
fig, ax = pf.create_plot()
ax.set_xlabel(r'$\log[\mathrm{R23}]$')
ax.set_ylabel(r'$\log[\mathrm{O23}]$')


ax1 = fig.add_axes([1,0,0.3,1], sharey=ax)
plt.setp(ax1.get_yticklabels(), visible=False)
ax2 = fig.add_axes([0,1,1,0.4], sharex=ax)
plt.setp(ax2.get_xticklabels(), visible=False)

cax = fig.add_axes([1.35,0,0.1,1])
cax.set_ylabel(r'$\log(EW\mathrm{[OIII]}_{5007})$')
cax.yaxis.set_label_position("right")

jiang_table = ascii.read('jiang_colorplot_data.csv')
O32_j = jiang_table['y']
R23_j = jiang_table['x']
cm = np.isfinite(tab['oiii5007_ew'])

ax.scatter(R23_j, O32_j, c='#77aca2', s=4, label='Jiang+ (2019)')
scatter = ax.scatter(np.log10(R23)[cm],np.log10(O32)[cm], s=30, marker='^',zorder=10, c=np.log10(tab['oiii5007_ew'])[cm], cmap='magma_r',vmax=3, vmin=.5, label='This Work')
cbar = fig.colorbar(scatter, ax=ax, cax=cax, label=r'$\log(EW\mathrm{[OIII]}_{5007})$')

xlims = np.array([0.5,1.25])
ylims = np.array([-0.6,1])

ax.set_xlim(xlims+0.06)
ax.set_ylim(ylims+0.06)
ax1.set_xticks([])
ax2.set_yticks([])
cax.set_ylim(0.6,2.5)


ax1.hist(O32_j, bins=np.linspace(0.6,1,15), density=True, orientation='horizontal', histtype='step', lw=2, ec='#77aca2')
ax2.hist(R23_j, bins=np.linspace(0.6,1.1,20), density=True, orientation='vertical', histtype='step', lw=2, ec='#77aca2')

ax1.hist(np.log10(O32), bins=np.linspace(-0.5,1,12), density=True, orientation='horizontal', histtype='step', lw=2, ec='#ff004f')
ax2.hist(np.log10(R23), bins=np.linspace(0.6,1.2,15), density=True, orientation='vertical', histtype='step', lw=2, ec='#ff004f')

ax.legend(loc='upper right', fontsize=12)

pf.fix_plot([ax,ax1,ax2,cax])

plt.tight_layout()
plt.savefig(f'figs/r23o32_{fixed}.png', dpi=600,bbox_inches='tight')


In [ ]:
plt.hist(tab['halpha_flux']/tab['hbeta_flux'], bins=np.linspace(1,5, 10))

In [ ]:
plt.hist(tab['hbeta_flux']/tab['hgamma_flux'], bins=np.linspace(-1,5, 10))

In [ ]:
fig, ax = pf.create_plot()
ax.hist(tab['Z_dir'], lw=2, density=True, edgecolor='#ff004f', hatch="/", facecolor='white')
pf.fix_plot([ax])
ax.set_yticks([])
ax.set_xlabel(r'$12+\log(\mathrm{O/H})$')
fig.savefig(f'figs/metallicity_{fixed}.png', dpi=600, bbox_inches='tight')

In [ ]:
z = tab['z']
D_L = cosmo.luminosity_distance(z)
D_L_cm = D_L.to(u.cm).value
L_hbeta = tab['hbeta_flux'] * 4 * np.pi * (D_L_cm**2)*1e-20
SFR = (7.9e-42*L_hbeta)*2.86 #kennicutt 1998 hbeta with halpha ratio

tab2 = Table(ascii.read('wpd_datasets.csv'), names=['c_x','c_y','b_x','b_y','d_x','d_y','e_x','e_y'])
cardamone = tab2['c_x'], tab2['c_y']
brunker = tab2['b_x'], tab2['b_y']
ding = tab2['d_x'], tab2['d_y']
eboss = tab2['e_x'], tab2['e_y']

datz = [cardamone,brunker,ding,eboss]
labz = ['Cardamone (2008)','Brunker (2020)', 'Ding (2025)', 'eBOSS Random Sample']
mkz = ['x','^','v','o']
colours = ['#9A48D0','#D95D39','#3E8989','k']
binz = [np.linspace(0,2,10),np.linspace(0.5,2,5), np.linspace(-2,2,10), np.linspace(-4,2,10)]

fig,ax= pf.create_plot()
ax1 = fig.add_axes([1,0,0.3,1], sharey=ax)
plt.setp(ax1.get_yticklabels(), visible=False)
ax2 = fig.add_axes([0,1,1,0.4], sharex=ax)
plt.setp(ax2.get_xticklabels(), visible=False)

mask = np.isfinite(np.log10(SFR))
ax.scatter(np.log10(SFR)[mask], np.log10(tab['oiii5007_ew'])[mask], label='This Work', marker='s', c='#ff004f', s=40)
ax1.hist( np.log10(tab['oiii5007_ew'])[mask], histtype='step', orientation='horizontal', lw=2, color='#ff004f', density=True, bins=10)
ax2.hist(np.log10(SFR)[mask], histtype='step', orientation='vertical', lw=2, color='#ff004f', density=True, bins=10)


for i,d in enumerate(datz):
    log10sfr_str = d[1][2:].data
    ew_str = d[0][2:].data 

    global dat
    dat = []

    for a in [log10sfr_str,ew_str]:
        if type(a) == np.ma.MaskedArray:
            a.set_fill_value(np.nan)
        dat.append(a.astype(np.float64))

    ax.scatter(dat[0],np.log10(dat[1]), label=labz[i], marker=mkz[i], s=(40 if i<3 else 4), color=colours[i])

    mask0 = np.isfinite(dat[0])
    mask1 = np.isfinite(dat[1])

    ax1.hist(np.log10(dat[1])[mask0], histtype='step', orientation='horizontal', lw=1, color=colours[i], density=True, bins=10)
    ax2.hist(dat[0][mask1],histtype='step', orientation='vertical', lw=1, color=colours[i], density=True, bins=10)
    
ax.set_ylim(0,4)
ax.set_xlim(-5.1,2.3)
ax.legend(fontsize=11)
ax.set_xlabel(r'$\log(\mathrm{SFR}\;[M_{\odot}/\mathrm{yr}])$')
ax.set_ylabel(r'$\log\;EW[\mathrm{[OIII]_{5007}}]$')

ax1.set_xticks([])
ax2.set_yticks([])

pf.fix_plot([ax,ax1,ax2])
fig.savefig(f'figs/SFR_{fixed}.png', dpi=600, bbox_inches='tight')


In [ ]:
fig, ax = pf.create_plot()

ax.set_xlabel(r'$\log$([N II]/H$\alpha$)')
ax.set_ylabel(r'$\log$([O III]/H$\beta$)')

xvals = np.log10(tab['nii6583_flux']/tab['halpha_flux'])
yvals = np.log10(tab['oiii5007_flux']/tab['hbeta_flux'])

xerr = np.abs(xvals*np.sqrt( (tab['nii6583_flux_err']/tab['nii6583_flux'])**2 + (tab['halpha_flux_err']/tab['halpha_flux'])**2 ))
yerr = np.abs(yvals*np.sqrt( (tab['oiii5007_flux_err']/tab['oiii5007_flux'])**2 + (tab['hbeta_flux_err']/tab['hbeta_flux'])**2 ))

classf = lr.classify_bpt(xvals,yvals)
markers = ['o','x','^']
colors = ['#ff004f', "#77aca2", "#0359c3"]


for i,key in enumerate(list(classf.keys())):
    mask = classf.get(key)
    ax.errorbar(xvals[mask],yvals[mask], fmt=markers[i], yerr=yerr[mask], color=colors[i], ms=4, ecolor='gray', capsize=2)
ax.set_xlim(-2.1,1.5)
ax.set_ylim(-1.5,1.5)

grid = np.linspace(-2.5,0, 100)
grid2 = np.linspace(-2.5,.3, 100)

ax.plot(grid,lr.kauffmann03(grid),color='k', ls='--')
ax.plot(grid2,lr.kewley01(grid2),color='k')

ax.text(-1.7,-1,'Starburst', fontsize=20)
ax.text(0.5,-1,'AGN', fontsize=20)

pf.fix_plot([ax])
fig.savefig(f'figs/bpt_{fixed}.png', dpi=600, bbox_inches='tight')

In [ ]:
fig, ax = pf.create_plot()
ax2 = fig.add_axes((1.1,0,0.1,1))

ax.set_xlabel(r'$\log$([N II]/H$\alpha$)')
ax.set_ylabel(r'$\log$([O III]/H$\beta$)')

xvals = np.log10(tab['nii6583_flux']/tab['halpha_flux'])
yvals = np.log10(tab['oiii5007_flux']/tab['hbeta_flux'])

xerr = np.abs(xvals*np.sqrt( (tab['nii6583_flux_err']/tab['nii6583_flux'])**2 + (tab['halpha_flux_err']/tab['halpha_flux'])**2 ))
yerr = np.abs(yvals*np.sqrt( (tab['oiii5007_flux_err']/tab['oiii5007_flux'])**2 + (tab['hbeta_flux_err']/tab['hbeta_flux'])**2 ))

classf = lr.classify_bpt(xvals,yvals)
markers = ['o','x','^']
colors = ['#ff004f', "#77aca2", "#0359c3"]


for i,key in enumerate(list(classf.keys())):
    mask = classf.get(key)

    oiiiEW = tab['oiii5007_ew'][mask]
    oiiiDISP = tab['oiii5007_vel_disp'][mask]
    # hbetaDISP = tab['hbeta_vel_disp'][mask]

    # ax.errorbar(xvals[mask],yvals[mask], fmt=markers[i], yerr=yerr[mask], color=colors[i], ms=4, ecolor='gray', capsize=2)
    ax.errorbar(xvals[mask],yvals[mask], fmt='none', yerr=yerr[mask], ms=4, ecolor='gray', capsize=2)
    sc = ax.scatter(xvals[mask],yvals[mask], marker = markers[i], c=np.log10(oiiiDISP), s=20, zorder=10, cmap='magma_r')
    fig.colorbar(sc, ax=ax,cax=ax2, label=r'log(V[OIII]$_{5007}$[ms$^{-1}$]) ')

ax.set_xlim(-2.1,1.5)
ax.set_ylim(-1.5,1.5)

grid = np.linspace(-2.5,0, 100)
grid2 = np.linspace(-2.5,.3, 100)

ax.plot(grid,lr.kauffmann03(grid),color='k', ls='--')
ax.plot(grid2,lr.kewley01(grid2),color='k')

ax.text(-1.7,-1,'Starburst', fontsize=20)
ax.text(0.5,-1,'AGN', fontsize=20)

pf.fix_plot([ax,ax2])
fig.savefig(f'figs/bpt_vel_{fixed}.png', dpi=600, bbox_inches='tight')

In [ ]:
fig, ax = pf.create_plot()
ax2 = fig.add_axes((1.1,0,0.1,1))

ax.set_xlabel(r'$\log$([N II]/H$\alpha$)')
ax.set_ylabel(r'$\log$([O III]/H$\beta$)')

xvals = np.log10(tab['nii6583_flux']/tab['halpha_flux'])
yvals = np.log10(tab['oiii5007_flux']/tab['hbeta_flux'])

xerr = np.abs(xvals*np.sqrt( (tab['nii6583_flux_err']/tab['nii6583_flux'])**2 + (tab['halpha_flux_err']/tab['halpha_flux'])**2 ))
yerr = np.abs(yvals*np.sqrt( (tab['oiii5007_flux_err']/tab['oiii5007_flux'])**2 + (tab['hbeta_flux_err']/tab['hbeta_flux'])**2 ))

classf = lr.classify_bpt(xvals,yvals)
markers = ['o','x','^']
colors = ['#ff004f', "#77aca2", "#0359c3"]


for i,key in enumerate(list(classf.keys())):
    mask = classf.get(key)

    oiiiEW = tab['neiii_ew'][mask]
    # oiiiDISP = tab['oiii5007_vel_disp'][mask]

    # ax.errorbar(xvals[mask],yvals[mask], fmt=markers[i], yerr=yerr[mask], color=colors[i], ms=4, ecolor='gray', capsize=2)
    ax.errorbar(xvals[mask],yvals[mask], fmt='none', yerr=yerr[mask], ms=4, ecolor='gray', capsize=2)
    sc = ax.scatter(xvals[mask],yvals[mask], marker = markers[i], c=np.log10(oiiiEW), s=20, zorder=10, cmap='cividis')
    fig.colorbar(sc, ax=ax,cax=ax2, label=r'log(EW[NeIII]) ')

ax.set_xlim(-2.1,1.5)
ax.set_ylim(-1.5,1.5)

grid = np.linspace(-2.5,0, 100)
grid2 = np.linspace(-2.5,.3, 100)

ax.plot(grid,lr.kauffmann03(grid),color='k', ls='--')
ax.plot(grid2,lr.kewley01(grid2),color='k')

ax.text(-1.7,-1,'Starburst', fontsize=20)
ax.text(0.5,-1,'AGN', fontsize=20)

pf.fix_plot([ax,ax2])
fig.savefig(f'figs/bpt_neiiiew_{fixed}.png', dpi=600, bbox_inches='tight')

In [ ]:
for i,key in enumerate(list(classf.keys())):
    mask = classf.get(key)

In [ ]:
fig, ax = pf.create_plot()

colors = ['#ff004f', "#E89844", "#a923e7"]

for i,key in enumerate(list(classf.keys())):
    mask = classf.get(key)
    ax.hist(tab['Z_dir'][mask], lw=2, density=True, edgecolor=colors[i], histtype='step', label=key, bins=np.linspace(7,12,10))

ax.hist(tab['Z_dir'], lw=2, density=True, edgecolor='k', facecolor="#77aca2", zorder=-1)
pf.fix_plot([ax])
ax.set_yticks([])
ax.set_xlabel(r'$12+\log(\mathrm{O/H})$')
plt.legend()
fig.savefig(f'figs/metallicitypops.png', dpi=600, bbox_inches='tight')

In [ ]:
fig, ax = pf.create_plot()

colors = ['#ff004f', "#E89844", "#a923e7"]

for i,key in enumerate(list(classf.keys())):
    mask = classf.get(key)
    ax.hist(np.log10(R23)[mask], lw=2, density=True, edgecolor=colors[i], histtype='step', label=key, bins=np.linspace(0.6,1.2,10))

ax.hist(np.log10(R23), lw=2, density=True, edgecolor='k', facecolor="#77aca2", zorder=-1,bins=np.linspace(0.6,1.2,10))
pf.fix_plot([ax])
ax.set_yticks([])
ax.set_xlabel(r'$\log(R23)$')
plt.legend()
fig.savefig(f'figs/r23pops_{fixed}.png', dpi=600, bbox_inches='tight')

In [ ]:
fig, ax = pf.create_plot()

colors = ['#ff004f', "#E89844", "#a923e7"]

for i,key in enumerate(list(classf.keys())):
    mask = classf.get(key)
    ax.hist(np.log10(O32)[mask], lw=2, density=True, edgecolor=colors[i], histtype='step', label=key, bins=np.linspace(-.5,1.1,10))

ax.hist(np.log10(O32), lw=2, density=True, edgecolor='k', facecolor="#77aca2", zorder=-1,bins=np.linspace(-.5,1.1,10))
pf.fix_plot([ax])
ax.set_yticks([])
ax.set_xlabel(r'$\log(O32)$')
plt.legend()
fig.savefig(f'figs/o32pops_{fixed}.png', dpi=600, bbox_inches='tight')

In [ ]:
fig, ax = pf.create_plot()
ax.set_xlabel(r'$\log[\mathrm{R23}]$')
ax.set_ylabel(r'$\log[\mathrm{O23}]$')


ax1 = fig.add_axes([1,0,0.3,1], sharey=ax)
plt.setp(ax1.get_yticklabels(), visible=False)
ax2 = fig.add_axes([0,1,1,0.4], sharex=ax)
plt.setp(ax2.get_xticklabels(), visible=False)

cax = fig.add_axes([1.35,0,0.1,1])
cax.set_ylabel(r'$\log(EW\mathrm{[OIII]}_{5007})$')
cax.yaxis.set_label_position("right")

jiang_table = ascii.read('jiang_colorplot_data.csv')
O32_j = jiang_table['y']
R23_j = jiang_table['x']
cm = np.isfinite(tab['oiii5007_ew'])

ax.scatter(R23_j, O32_j, c='#77aca2', s=4, label='Jiang+ (2019)')
scatter = ax.scatter(np.log10(R23)[cm],np.log10(O32)[cm], s=30, marker='^',zorder=10, c=np.log10(tab['oiii5007_ew'])[cm], cmap='magma_r',vmax=3, vmin=.5, label='This Work')
cbar = fig.colorbar(scatter, ax=ax, cax=cax, label=r'$\log(EW\mathrm{[OIII]}_{5007})$')

colors = ['#ff004f', "#E89844", "#a923e7"]
for i,key in enumerate(list(classf.keys())):
    # mask = (classf.get(key))&(xvals<-1.8)
    mask = (classf.get(key))
    scatter = ax.scatter(np.log10(R23)[mask],np.log10(O32)[mask], s=80, marker='s', c='none', edgecolors=colors[i])


xlims = np.array([0.5,1.25])
ylims = np.array([-0.6,1])

ax.set_xlim(xlims+0.06)
ax.set_ylim(ylims+0.06)
ax1.set_xticks([])
ax2.set_yticks([])
cax.set_ylim(0.6,2.5)


ax1.hist(O32_j, bins=np.linspace(0.6,1,15), density=True, orientation='horizontal', histtype='step', lw=2, ec='#77aca2')
ax2.hist(R23_j, bins=np.linspace(0.6,1.1,20), density=True, orientation='vertical', histtype='step', lw=2, ec='#77aca2')

ax1.hist(np.log10(O32), bins=np.linspace(-0.5,1,12), density=True, orientation='horizontal', histtype='step', lw=2, ec='#ff004f')
ax2.hist(np.log10(R23), bins=np.linspace(0.6,1.2,15), density=True, orientation='vertical', histtype='step', lw=2, ec='#ff004f')

ax.legend(loc='upper right', fontsize=12)

pf.fix_plot([ax,ax1,ax2,cax])

plt.tight_layout()
plt.savefig(f'figs/r23o32_layered_{fixed}.png', dpi=600,bbox_inches='tight')

In [ ]:
foreground = tab.group_by('foreground').groups[1]
cluster = tab.group_by('cluster_member').groups[1]
lensed = tab.group_by('lensed').groups[1]

groups = [foreground,cluster,lensed]
names = ['Foreground','Cluster Members','Lensed']

fig,ax=pf.create_plot(size=(3,3))
ax2 = fig.add_axes((1.2,0,1,1))

ax.set_yticks([])
ax2.set_yticks([])

ax.hist(tab['z']-tab['zcluster'], density=True, edgecolor='k',facecolor="#6F6F6F")
ax2.hist(tab['z'], density=True, edgecolor='k',facecolor="#6F6F6F")

for i,g in enumerate(groups):
    ax.hist(g['z']-g['zcluster'], density=True, histtype='step', lw=2, label=names[i])
    ax2.hist(g['z'], density=True, histtype='step', lw=2)

ax.set_xlabel(r'$z-z_{\mathrm{cluster}}$')
ax2.set_xlabel(r'$z$')

ax.legend()
pf.fix_plot([ax,ax2])
plt.savefig(f'figs/redshift_dist_{fixed}.png', dpi=600,bbox_inches='tight')

In [ ]:
fig, ax = pf.create_plot()
ax2 = fig.add_axes((1.1,0,0.1,1))

ax.set_xlabel(r'$\log$([N II]/H$\alpha$)')
ax.set_ylabel(r'$\log$([O III]/H$\beta$)')

xvals = np.log10(tab['nii6583_flux']/tab['halpha_flux'])
yvals = np.log10(tab['oiii5007_flux']/tab['hbeta_flux'])

xerr = np.abs(xvals*np.sqrt( (tab['nii6583_flux_err']/tab['nii6583_flux'])**2 + (tab['halpha_flux_err']/tab['halpha_flux'])**2 ))
yerr = np.abs(yvals*np.sqrt( (tab['oiii5007_flux_err']/tab['oiii5007_flux'])**2 + (tab['hbeta_flux_err']/tab['hbeta_flux'])**2 ))

classf = lr.classify_bpt(xvals,yvals)
markers = ['o','x','^']
colors = ['#ff004f', "#77aca2", "#0359c3"]


for i,key in enumerate(list(classf.keys())):
    mask = classf.get(key)

    # oiiiEW = tab['oiii5007_ew'][mask]
    # oiiiDISP = tab['oiii5007_vel_disp'][mask]
    # hbetaDISP = tab['hbeta_vel_disp'][mask]
    O32DISP = R23[mask]

    # ax.errorbar(xvals[mask],yvals[mask], fmt=markers[i], yerr=yerr[mask], color=colors[i], ms=4, ecolor='gray', capsize=2)
    ax.errorbar(xvals[mask],yvals[mask], fmt='none', yerr=yerr[mask], ms=4, ecolor='gray', capsize=2)
    sc = ax.scatter(xvals[mask],yvals[mask], marker = markers[i], c=np.log10(O32DISP), s=20, zorder=10, cmap='magma_r')
    fig.colorbar(sc, ax=ax,cax=ax2, label=r'log(R23) ')

ax.set_xlim(-2.1,1.5)
ax.set_ylim(-1.5,1.5)

grid = np.linspace(-2.5,0, 100)
grid2 = np.linspace(-2.5,.3, 100)

ax.plot(grid,lr.kauffmann03(grid),color='k', ls='--')
ax.plot(grid2,lr.kewley01(grid2),color='k')

ax.text(-1.7,-1,'Starburst', fontsize=20)
ax.text(0.5,-1,'AGN', fontsize=20)

pf.fix_plot([ax,ax2])
fig.savefig(f'figs/bpt_R23_{fixed}.png', dpi=600, bbox_inches='tight')

In [ ]:
ratio = tab['oiii5007_flux']/tab['oiii4959_flux']
mask1 = np.isfinite(ratio)
mask2 = classf.get('starburst')

fig, ax = pf.create_plot()

ax.hist(ratio[mask1&mask2], bins = np.linspace(0,8,30))
# ax.scatter(tab['oiii5007_flux'],tab['oiii4959_flux'])

pf.fix_plot([ax])

In [ ]:
fig, ax = pf.create_plot()
mask3 = classf.get('starburst')
ax.scatter(xvals[mask3], np.log10(R23)[mask3])
pf.fix_plot([ax])

In [ ]:
subtab = tab[classf.get('agn')].copy()
agns = Table([subtab['ra'], subtab['dec'], subtab['z'], subtab['name']])
agns.write('candidate_agns.csv', overwrite=True)

In [ ]:
tab

In [ ]:
plt.hist(tab['oiii5007_ew'], bins=np.linspace(0,1000))

In [ ]:
fig, ax = pf.create_plot()
ax.set_xlabel(r'$\log[\mathrm{R23}]$')
ax.set_ylabel(r'$\log[\mathrm{O23}]$')


ax1 = fig.add_axes([1,0,0.3,1], sharey=ax)
plt.setp(ax1.get_yticklabels(), visible=False)
ax2 = fig.add_axes([0,1,1,0.4], sharex=ax)
plt.setp(ax2.get_xticklabels(), visible=False)

cax = fig.add_axes([1.35,0,0.1,1])
cax.set_ylabel(r'$\log(EW\mathrm{[OIII]}_{5007})$')
cax.yaxis.set_label_position("right")

jiang_table = ascii.read('jiang_colorplot_data.csv')
O32_j = jiang_table['y']
R23_j = jiang_table['x']
cm = np.isfinite(tab['oiii5007_ew'])#&((tab['oiii5007_ew']>300)|(tab['hbeta_ew']>100))&((tab['oiii4363_flux']/np.abs(tab['oiii4363_flux_err']))>3)

ax.scatter(R23_j, O32_j, c='#77aca2', s=4, label='Jiang+ (2019)')
scatter = ax.scatter(np.log10(R23)[cm],np.log10(O32)[cm], s=30, marker='^',zorder=10, c=np.log10(tab['oiii5007_ew'])[cm], cmap='magma_r',vmax=3, vmin=.5, label='This Work')
cbar = fig.colorbar(scatter, ax=ax, cax=cax, label=r'$\log(EW\mathrm{[OIII]}_{5007})$')

colors = ['#ff004f', "#E89844", "#a923e7"]
for i,key in enumerate(list(classf.keys())):
    # mask = (classf.get(key))&(xvals<-1.8)
    mask = (classf.get(key))&cm
    scatter = ax.scatter(np.log10(R23)[mask],np.log10(O32)[mask], s=80, marker='s', c='none', edgecolors=colors[i])


xlims = np.array([0.5,1.25])
ylims = np.array([-0.6,1])

ax.set_xlim(xlims+0.06)
ax.set_ylim(ylims+0.06)
ax1.set_xticks([])
ax2.set_yticks([])
cax.set_ylim(0.6,2.5)


ax1.hist(O32_j, bins=np.linspace(0.6,1,15), density=True, orientation='horizontal', histtype='step', lw=2, ec='#77aca2')
ax2.hist(R23_j, bins=np.linspace(0.6,1.1,20), density=True, orientation='vertical', histtype='step', lw=2, ec='#77aca2')

ax1.hist(np.log10(O32), bins=np.linspace(-0.5,1,12), density=True, orientation='horizontal', histtype='step', lw=2, ec='#ff004f')
ax2.hist(np.log10(R23), bins=np.linspace(0.6,1.2,15), density=True, orientation='vertical', histtype='step', lw=2, ec='#ff004f')

poptable = ascii.read('poptable.csv')
cm2 = np.isfinite(poptable['oiiiEW'])#&((poptable['oiiiEW']>300)|(poptable['hbetaEW']>100))
ax.scatter(poptable['log10R23'][cm2], poptable['log10O32'][cm2],marker='x', s=10, c=np.log10(poptable['oiiiEW'])[cm2],cmap='magma_r',vmax=3, vmin=.5,label='MUSE-WIDE')
ax.errorbar(poptable['log10R23'][cm2],poptable['log10O32'][cm2],xerr=poptable['log10R23_e'][cm2]/10, yerr=poptable['log10O32_e'][cm2]/10, fmt='none', ecolor='gray', capsize=2, zorder=-10, alpha=0.4)

ax1.hist(poptable['log10O32'], bins=np.linspace(-0.5,1,12), density=True, orientation='horizontal', histtype='step', lw=2, ec='k')
ax2.hist(poptable['log10R23'], bins=np.linspace(0.6,1.2,15), density=True, orientation='vertical', histtype='step', lw=2, ec='k')

legend_elements = [
    Line2D([0], [0], marker='o', color='k', label='Jiang+19',
           markerfacecolor='#77aca2', markersize=10),
    Line2D([0], [0], marker='^', color='k', label='This Work',
           markerfacecolor='#ff004f', markersize=10),
    Line2D([0], [0], marker='x', color='k', label='MUSE-WIDE',
           markerfacecolor='#ff004f', markersize=10)
]


ax.legend(loc='upper right', fontsize=12, handles=legend_elements)

pf.fix_plot([ax,ax1,ax2,cax])

plt.tight_layout()
plt.savefig(f'figs/r23o32_layered_POP_{fixed}.png', dpi=600,bbox_inches='tight')

In [ ]:
# newtable = process_table(tab)

In [1]:
tab

NameError: name 'tab' is not defined